In [ ]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[3, 5, 8, 10, 13])

nb_name = "Hotspot.ipynb"

_compare_version(nb_name)

# Hotspot
<hr style="border:2px solid black"> </hr>

## 1 - Description

# Hotspot Analysis

Hotspot identifies genes with informative local expression structure in single-cell data by computing gene-wise autocorrelations on a cell neighborhood graph. Based on these genes, local gene-gene correlations can be calculated and grouped into modules representing putative gene programs.
Hotspot documentation: https://hotspot.readthedocs.io/en/latest/

The aim of this notebook is to detect locally variable genes, calculate local gene-gene correlations, and group correlated genes into modules. These modules can represent putative gene programs that are active in specific cell states, developmental stages, or cell populations.

The main outputs of this notebook are Hotspot autocorrelation results, selected informative genes, local correlation matrices, gene modules, module scores per cell, summary tables, and visualizations such as UMAP plots and module-level summaries.

## Prerequisites

Requires a preprocessed AnnData object with raw counts stored in a layer, usually `adata.layers["raw"]`, and relevant metadata in `adata.obs` for plotting or subsetting.

This notebook requires a low-dimensional representation of the data to construct the cell neighborhood graph used by Hotspot. PCA and Harmony-corrected PCA can be computed within this notebook if needed. Other latent representations, such as scVI or Scanorama, must already be present in `adata.obsm` if they should be used for Hotspot.

Move this notebook into the notebook folder of the respective analysis, for example `rna_analysis/notebooks/`, before using it.


_______

## 2 - Setup

In [ ]:
import sctoolbox
import sctoolbox.utils as utils
import sctoolbox.tools.hotspot as hotspot
import sctoolbox.plotting as pl

import pandas as pd
from sctoolbox import settings

settings.settings_from_config("config.yaml", key="hotspot")

with pd.option_context("display.max.rows", None, "display.max_colwidth", None):
    display(utils.general.get_version_report(report="versions.yml"))

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Set inpput anndata
clustered_adata = "anndata_4.h5ad"

_________

## 3 - Load anndata

In [ ]:
adata = utils.adata.load_h5ad(clustered_adata)

with pd.option_context("display.max.rows", 5, "display.max.columns", None):
    display(adata)
    display(adata.obs)
    display(adata.var)

## 4 -  Plot UMAP
<hr style="border:2px solid black"> </hr>

This section visualizes the full AnnData object before optional subsetting and Hotspot analysis. The global UMAP provides an overview of the dataset structure and shows how cells are distributed according to the selected condition column.

This helps users inspect sample, condition, batch, or cell population patterns before deciding whether to run Hotspot on the full dataset or on a biologically relevant subset. The UMAP is used for exploratory visualization and does not determine the latent representation used by Hotspot later in the notebook.


<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Global UMAP overview before subsetting

# Column in adata.obs used to color the UMAP plot.
condition_column = "sample"

# Low-dimensional representation used to compute the UMAP if needed.
# Common options are "X_pca", "X_pca_harmony", "X_scVI", or "X_scanorama".
global_umap_use_rep = "X_pca"

# If True, recompute the UMAP even if adata.obsm["X_umap"] already exists.
global_force_recompute_umap = False

# If True, recompute the representation used for UMAP, e.g. PCA or Harmony.
# This only matters if the selected representation is missing or should be overwritten.
global_force_recompute_rep = False

# Shared UMAP computation settings.
# These parameters are passed to the UMAP computation step.
umap_kwargs = {
    "min_dist": 0.5,        # Controls how tightly cells are packed together.
    "spread": 1.0,          # Controls the overall spread of the UMAP layout.
    "n_components": 2,      # Number of UMAP dimensions to compute.
    "random_state": 0,      # Random seed for reproducible UMAP results.
}

# Shared plotting settings for UMAP visualizations.
# These parameters are passed to pl.embedding.plot_embedding().
umap_plot_kwargs = {
    "method": "umap",       # Plot the UMAP embedding stored in adata.obsm["X_umap"].
    "color": condition_column,  # Color cells by the selected metadata column.
    "ncols": 3,             # Number of plot columns if multiple colors are shown.
    "rasterize": True,      # Rasterize points when saving to reduce file size.
    "title": f"UMAP\ncolored by {condition_column}",  # Plot title.
}

In [ ]:
hotspot.ensure_umap_for_visualization(
    adata,
    use_rep=global_umap_use_rep,
    is_subset=False,
    recompute_if_subset=False,
    force_recompute_umap=global_force_recompute_umap,
    force_recompute_rep=global_force_recompute_rep,
    neighbors_key="neighbors_global_umap",
    **umap_kwargs,
)

_ = pl.embedding.plot_embedding(
    adata,
    save="hotspot_global_umap.pdf",
    report="hotspot_global_umap.png",
    **umap_plot_kwargs,
)

__________

## 5 - Subset data (optional)
<hr style="border:2px solid black"> </hr>


### Why should I subset?

Hotspot groups genes by how their expression varies across the cell
neighbourhood graph. On a full dataset the strongest such variation is almost
always the difference between cell types, so the resulting modules tend to be
dominated by cell type marker genes and mostly reproduce the clustering you
already have.

Subsetting to a biologically relevant population removes that dominant signal
and lets finer programs surface: variation *within* one cell type, such as cell
cycle, differentiation gradients, stress response or treatment effects.

**Rule of thumb:** subset when you have a specific biological question about a
population. Run on the full dataset only when you want a broad overview and
accept that the modules will largely follow cell types.

Examples of a relevant subset: one cell type (`celltype == "Epithelial"`), one
condition, one timepoint, or a combination of these.

If you subset here, PCA/Harmony, the neighbour graph and the UMAP are
recomputed in sections 7 and 8, since the existing embeddings were computed on
all cells and would be skewed for the subset.

An empty dictionary (`{}`) means no subsetting; the object is left unchanged.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# If empty the Anndata won't be changed
# Example:
# subset_filters = {
#     "sample": [3, 6],
#     "subcelltype": ["A+", "A-"],
# }

subset_filters = {}

# If True, create a copy of the subsetted AnnData object.
# This avoids modifying the original AnnData object in place.
subset_copy = True

# If True, remove unused categories from categorical obs columns after subsetting.
# Set to False to keep the original category structure.
drop_unused_categories = False

In [ ]:
# Subset the celltype an early stage for example
adata_used = utils.adata.obs_subset(
        adata=adata,
        subset_filters=subset_filters,
        copy=subset_copy,
        drop_unused_categories=drop_unused_categories,
    )
adata_used

In [ ]:
# Set subset true
subset_was_applied = utils.adata.was_subsetted(
    original=adata,
    current=adata_used,
)

print(f"Subset was applied: {subset_was_applied}")

## 6 - Filter genes
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Gene detection filtering settings
GENE_DETECTION_FILTERING = {
    # Layer used to count in how many cells each gene is detected.
    # Raw/count-like data are recommended, e.g. "raw" or "counts".
    # Set to None to use adata.X.
    "layer_key": "raw",

    # Minimum number of cells in which a gene must be detected.
    # Genes seen in very few cells carry little information for the
    # neighbourhood statistics and mostly add noise, so removing them can
    # sharpen the Hotspot result. As a rule of thumb, start at roughly
    # 0.1-1% of the cells in the current (sub)set and check the histogram
    # and the threshold table printed by this function before deciding.
    # The default of 50 fits datasets of a few thousand to a few tens of
    # thousands of cells; for smaller subsets a lower value is usually
    # more appropriate. With filter_genes=False nothing is removed and
    # this value only affects the reported summary.
    "min_cells": 50,

    # Only inspect thresholds, useful for deciding min_cells
    "thresholds_to_show": [10, 20, 30, 50, 100],

    # If False, only show statistics and plot.
    # If True, actually remove genes detected in fewer than min_cells cells.
    "filter_genes": True,

    "recompute": subset_was_applied,
}

In [ ]:
adata_filtered, gene_detection, gene_filter_summary = hotspot.inspect_and_filter_genes_by_cell_detection(
    adata_used,
    **GENE_DETECTION_FILTERING,
)

## 7 - Select and calculate Hotspot latent key for hotspot
<hr style="border:2px solid black"> </hr>

### What is a latent representation?

Hotspot asks, for every gene, whether cells that are close to each other also
show similar expression of that gene. "Close" is defined by a KNN graph, and
that graph is not built on the raw genes but on a low-dimensional summary of
the cells — the latent representation. Each cell is described by a few
dimensions (e.g. 50 principal components) that capture the main variation in
the dataset, which removes most of the technical noise and makes distances
between cells meaningful.

### What happens in this step?

The notebook selects one representation from `adata.obsm` and stores its key in
`hotspot_latent_key`. PCA and Harmony-corrected PCA are computed here if they
are missing. Representations from other tools, such as `X_scVI` or
`X_scanorama`, are reused if they already exist but are not computed by this
notebook.

| Representation | When to use it |
|:---|:---|
| `X_pca` | Default. One batch, or batch effects negligible. |
| `X_pca_harmony` | Several samples or batches that should be corrected. |
| `X_scVI`, `X_scanorama` | An integration already computed in an earlier notebook. |

With `preferred_rep = "auto"` the entries in `priority` are tried in order and
the first usable one is taken. With an explicit key that representation is
enforced and any problem raises an error.

### Why does this matter?

The representation decides which cells count as neighbours, and therefore the
entire result. A representation that still carries a batch effect produces
modules that follow the batch rather than biology. One that is too coarse
smooths away the programs you are looking for.

Note that this is separate from the UMAP in sections 4 and 8: the UMAP is for
looking at the data, the latent representation is what Hotspot computes on.

### Further reading

- Hotspot documentation: https://hotspot.readthedocs.io/en/latest/
- DeTomaso & Yosef, *Hotspot identifies informative gene modules across
  modalities of single-cell genomics*, Cell Systems 12(5), 2021:
  https://doi.org/10.1016/j.cels.2021.04.005
- Harmony (batch correction): Korsunsky et al., *Fast, sensitive and accurate
  integration of single-cell data with Harmony*, Nature Methods 16, 2019:
  https://doi.org/10.1038/s41592-019-0619-0
  

### Parameters

#### Metadata columns

| Parameter | Description | Options / Example | Default |
|:---|:---|:---|:---|
| `condition_column` | Column in `adata.obs` used for plotting and colouring cells. | String, e.g. `"sample"`, `"cell_type"` | `"sample"` |
| `batch_key` | Column in `adata.obs` used for Harmony batch correction. `None` disables batch correction. | `None` or String, e.g. `"sample"`, `"batch"` | `"sample"` |

#### `HOTSPOT_REPRESENTATION` — which representation to use

| Key | Description | Options | Default |
|:---|:---|:---|:---|
| `preferred_rep` | Representation Hotspot builds its neighbour graph on. `"auto"` picks the first usable entry from `priority`. | `"auto"`, `"X_pca"`, `"X_pca_harmony"`, `"X_scVI"`, `"X_scanorama"` | `"auto"` |
| `priority` | Order in which representations are tried when `preferred_rep="auto"`. | List of representation keys | `["X_pca_harmony", "X_scVI", "X_scanorama", "X_pca"]` |
| `compute_pca_if_missing` | Whether PCA may be computed if it is not present. | `True`, `False` | `True` |
| `compute_harmony_if_missing` | Whether Harmony may be computed if it is not present and a valid `batch_key` is available. | `True`, `False` | `True` |
| `recompute_if_subset` | Whether PCA and Harmony are recomputed after subsetting. Recommended, since existing embeddings describe all cells. | `True`, `False` | `True` |
| `force_recompute_pca` | Whether an existing PCA is overwritten, regardless of subsetting. | `True`, `False` | `False` |
| `force_recompute_harmony` | Whether an existing Harmony representation is overwritten, regardless of subsetting. | `True`, `False` | `False` |
| `allow_existing_external_after_subset` | Whether `X_scVI` or `X_scanorama` may still be used after subsetting. They are not recomputed by this notebook, so they still describe all cells. | `True`, `False` | `False` |

#### `PCA_SETTINGS` — how PCA is computed

| Key | Description | Options | Default |
|:---|:---|:---|:---|
| `pca_key` | Key the PCA is stored under in `adata.obsm`. | String | `"X_pca"` |
| `n_comps` | Number of principal components. More components keep finer structure but also more noise. | Integer | `50` |
| `use_highly_variable` | Whether to restrict PCA to highly variable genes. Requires `adata.var["highly_variable"]`. | `True`, `False` | `False` |
| `layer` | Layer used for the PCA. `None` uses `adata.X`. | `None` or String, e.g. `"raw"` | `None` |
| `svd_solver` | SVD solver passed to Scanpy. `None` lets Scanpy choose. | `None` or String | `None` |
| `random_state` | Random seed for reproducible results. | Integer | `0` |

#### `HARMONY_SETTINGS` — how Harmony is computed

| Key | Description | Options | Default |
|:---|:---|:---|:---|
| `basis` | Representation Harmony corrects. | String | `"X_pca"` |
| `adjusted_basis` | Key the corrected representation is stored under in `adata.obsm`. | String | `"X_pca_harmony"` |
| `batch_key` | Column Harmony corrects for. Usually taken from `batch_key` above. | `None` or String | `batch_key` |

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Metadata columns
condition_column = "sample"      # for plotting / coloring
batch_key = "sample"

HOTSPOT_REPRESENTATION = {
    # Options:
    # "auto"           -> use next available representation from priority
    # "X_pca"          -> compute/use PCA
    # "X_pca_harmony"  -> compute/use Harmony-corrected PCA
    # "X_scVI"         -> use existing scVI representation if available
    # "X_scanorama"    -> use existing Scanorama representation if available
    "preferred_rep": "auto",

    # Priority if preferred_rep == "auto"
    "priority": [
        "X_pca_harmony",
        "X_scVI",
        "X_scanorama",
        "X_pca",
    ],

    "compute_pca_if_missing": True,
    "compute_harmony_if_missing": True,
    "recompute_if_subset": True,
    "force_recompute_pca": False,
    "force_recompute_harmony": False,
    "allow_existing_external_after_subset": False,
}

# PCA settings
PCA_SETTINGS = {
    "pca_key": "X_pca",
    "n_comps": 50,
    "use_highly_variable": False,
    "layer": None,
    "svd_solver": None,
    "random_state": 0,
}

# Harmony settings
HARMONY_SETTINGS = {
    "basis": "X_pca",
    "adjusted_basis": "X_pca_harmony",
    "batch_key": batch_key,
}

# UMAP visualization setting after optional subsetting
subset_umap_use_rep = "X_pca"

In [ ]:
hotspot_latent_key = hotspot.ensure_hotspot_latent_representation(
    adata=adata_filtered,
    representation_settings=HOTSPOT_REPRESENTATION,
    pca_settings=PCA_SETTINGS,
    harmony_settings=HARMONY_SETTINGS,
    is_subset=subset_was_applied,
)

print(f"Selected Hotspot latent representation: {hotspot_latent_key}")

## 8 - (Re-) Calculate UMAP after subsetting
<hr style="border:2px solid black"> </hr>


### Why is the UMAP recomputed?

The UMAP from section 4 was computed on all cells. If you subsetted in
section 5, that layout no longer describes the data you are working with: its
axes are still dominated by the populations you removed, and the differences
that remain inside your subset get compressed into a small part of the plot.

Recomputing gives the subset its own layout, so that structure within the
selected population becomes visible — which is exactly the structure Hotspot
will later report as modules.

Without subsetting an existing UMAP is reused and nothing is overwritten.

### What is this UMAP used for?

Visualisation only. The module scores in section 10 are plotted on it, which
makes it easier to see where in the data a module is active.

The neighbour graph Hotspot computes on is built separately from
`hotspot_latent_key` (sections 7 and 9). The two can use different
representations, and changing `subset_umap_use_rep` does not change the
Hotspot result.




### Parameters

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `subset_recompute_if_subset` | Whether PCA/Harmony and UMAP are recomputed when the data was subsetted in section 5. Leave `True` unless you deliberately want to keep the full-dataset layout. | `True`, `False` | `True` |
| `subset_force_recompute_umap` | Recompute the UMAP even without subsetting, e.g. after changing the UMAP parameters. | `True`, `False` | `False` |
| `subset_force_recompute_rep` | Recompute the representation the UMAP is built on, e.g. after changing `n_comps`. | `True`, `False` | `False` |
| `subset_neighbors_key` | Key the neighbour graph for this UMAP is stored under in `adata.uns`. A separate key keeps it apart from the graph of section 4. | String | `"neighbors_hotspot_umap"` |
| `umap_plot_kwargs["color"]` | Column in `adata.obs` used to colour the cells. | String or list of strings | `condition_column` |
| `umap_plot_kwargs["ncols"]` | Number of plot columns when several colours are shown. | Integer | `3` |
| `umap_plot_kwargs["rasterize"]` | Rasterise the points when saving, which keeps the PDF small on large datasets. | `True`, `False` | `True` |

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Whether to recompute the UMAP representation after subsetting.
# If True, PCA/Harmony and UMAP can be recomputed for the subset.
subset_recompute_if_subset = True

# If True, recompute the UMAP even if adata_filtered.obsm["X_umap"] already exists.
subset_force_recompute_umap = False

# If True, recompute the representation used for UMAP, e.g. PCA or Harmony.
subset_force_recompute_rep = False

# Key used to store the neighbor graph for the subset UMAP.
subset_neighbors_key = "neighbors_hotspot_umap"

# Shared plotting settings for the subset UMAP.
# These settings are passed to pl.embedding.plot_embedding().
umap_plot_kwargs = {
    "method": "umap",  # Plot the UMAP embedding stored in adata.obsm["X_umap"].
    "color": condition_column,  # Color cells by the selected metadata column.
    "ncols": 3,  # Number of plot columns if multiple colors are shown.
    "rasterize": True,  # Rasterize points when saving to reduce file size.
    "title": f"UMAP after subsetting\ncolored by {condition_column}",
}

In [ ]:
hotspot.ensure_umap_for_visualization(
    adata_filtered,
    use_rep=subset_umap_use_rep,
    batch_key=batch_key,
    is_subset=subset_was_applied,
    recompute_if_subset=subset_recompute_if_subset,
    force_recompute_umap=subset_force_recompute_umap,
    force_recompute_rep=subset_force_recompute_rep,
    neighbors_key=subset_neighbors_key,
    use_highly_variable=PCA_SETTINGS["use_highly_variable"],
    **umap_kwargs,
)

_ = pl.embedding.plot_embedding(
    adata_filtered,
    save="hotspot_subset_umap.pdf",
    report="hotspot_subset_umap.png",
    **umap_plot_kwargs,
)

<h1><center>⬐ Fill in input data here ⬎</center></h1>

## 9 - Hotspot Run
<hr style="border:2px solid black"> </hr>

### Where to start

If you are unsure, the defaults below are a reasonable first run. The three
parameters worth revisiting afterwards:

- **`n_neighbors`** controls how far Hotspot looks. Too small and modules
  fragment into noise, too large and distinct programs get merged. Start at
  `30`, lower it for small subsets.
- **`gene_selection_method`** decides how many genes enter the correlation
  step, which drives the runtime. `"top_n_z"` with 500 genes is fast and
  usually enough to see the main programs. Switch to `"fdr"` once you know the
  dataset and want completeness.
- **`min_gene_threshold`** sets how granular the modules are. If you get two
  or three very large modules, lower it; if you get many tiny ones, raise it.

None of these has one correct value — run once, look at the module sizes and
the score UMAPs in section 10, and adjust.

### Parameters

Hotspot is not limited to scRNA-seq. For other modalities the layer and the
null model have to match the data — see `model` below and the
[Hotspot documentation](https://hotspot.readthedocs.io/en/latest/).

#### Input

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `layer_key` | Layer Hotspot reads. Should hold raw counts, not normalised values. For scATAC this is the peak matrix. | String, e.g. `"raw"`, `"counts"` | `"raw"` |
| `latent_obsm_key` | Representation the neighbour graph is built on, selected in section 7. | `hotspot_latent_key`, `"X_pca"`, `"X_pca_harmony"`, `"X_scVI"`, `"X_scanorama"` | `hotspot_latent_key` |
| `umi_counts_obs_key` | Column in `adata.obs` with the total counts per cell. Computed from `layer_key` if missing. | String | `"total_counts"` |
| `model` | Null model for the per-gene distribution. Must match what is in `layer_key`: `"danb"` for UMI counts (scRNA-seq), `"bernoulli"` for binary or very sparse data such as scATAC peak matrices, `"normal"` for continuous normalised data, `"none"` for already variance-stabilised data. [Details](https://hotspot.readthedocs.io/en/latest/) | `"danb"`, `"bernoulli"`, `"normal"`, `"none"` | `"danb"` |

#### Neighbour graph

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `n_neighbors` | Number of neighbouring cells. Small values (10–15) pick up fine, local programs but are noisier; large values (50–100) give smoother, broader programs. For subsets below ~1000 cells use 10–15. | Integer | `30` |
| `weighted_graph` | Whether neighbours are weighted by distance. `False` treats all neighbours equally and is more robust. | `True`, `False` | `False` |
| `jobs` | Number of parallel jobs. Taken from `config.yaml`. | Integer | `settings.threads` |

#### Gene selection

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `gene_selection_method` | How genes for the local correlations are chosen. `"top_n_z"` takes a fixed number and keeps the runtime predictable; `"fdr"` is data-driven but can select many thousands of genes on large datasets. Start with `"top_n_z"`. | `"top_n_z"`, `"fdr"` | `"top_n_z"` |
| `top_n` | Number of genes taken by Z-score. Only used with `"top_n_z"`. | Integer, 500–1000 to start | `500` |
| `autocorr_fdr_threshold` | FDR cutoff for gene selection. Only used with `"fdr"`. | Float | `0.05` |

#### Modules

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `min_gene_threshold` | Minimum number of genes per module. Smaller values give more and smaller modules. | Integer | `20` |
| `core_only` | Hotspot builds modules by hierarchically clustering the locally correlated genes. A gene counts as a **core gene** of a module when it correlates strongly with the module as a whole, not just with a few of its members. With `True` only those genes are assigned and the rest is labelled unassigned (`-1`), which gives smaller but cleaner modules. With `False` genes at the cluster border are assigned as well, giving larger modules with a less specific signature. Keep `True` unless you deliberately want broader gene lists, e.g. for enrichment. | `True`, `False` | `True` |
| `module_fdr_threshold` | FDR cutoff used when modules are created. | Float | `0.05` |

#### Which steps to run

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `compute_local_correlations` | Compute gene-gene correlations for the selected genes. Required for modules. | `True`, `False` | `True` |
| `compute_modules` | Group correlated genes into modules. | `True`, `False` | `True` |
| `compute_module_scores` | Compute a per-cell score for each module. Requires `compute_modules=True`. | `True`, `False` | `True` |

#### Output

| Parameter | Description | Options | Default |
|:---|:---|:---|:---|
| `module_prefix` | Prefix of the module score columns added to `adata.obs`. | String | `"Hotspot module"` |
| `store_local_correlations` | Whether to keep the local correlation matrix in `adata.uns`. It grows with the square of the number of selected genes, so the `.h5ad` gets large. | `True`, `False` | `False` |

### 9.1 - input hotspot parameters

In [ ]:
# Hotspot input settings
umi_counts_obs_key = "total_counts"

# Null model for the per-gene expression distribution.
# Pick the one that matches the input in layer_key:
#   "danb"      - depth-adjusted negative binomial. For UMI counts, i.e. the
#                 usual choice for scRNA-seq with layer_key="raw".
#   "bernoulli" - models detection (zero vs non-zero) only. For binary or very
#                 sparse data such as scATAC peak matrices, or when counts are
#                 unreliable.
#   "normal"    - for continuous data that is already normalised and
#                 log-transformed, or for non-count modalities.
#   "none"      - no null model; values are only centered and scaled. Use when
#                 the data is already variance-stabilised.
# See https://hotspot.readthedocs.io/en/latest/ for the underlying statistics.
model = "danb"

# Graph settings
n_neighbors = 30
weighted_graph = False
jobs = settings.threads

# Gene selection
gene_selection_method = "top_n_z"   # available: "fdr", "top_n_z"
autocorr_fdr_threshold = 0.05
top_n = 500

# Module settings
min_gene_threshold = 20
core_only = True
module_fdr_threshold = 0.05

# Outputs
compute_local_correlations = True
compute_modules = True
compute_module_scores = True

module_prefix = "Hotspot module"

# Whether to keep the local correlation matrix in adata.uns.
# It grows with the square of the number of selected genes;
# keep False unless you need it, otherwise the .h5ad gets large.
store_local_correlations = False

### 9.2 - Run hotspot

In [ ]:
results = hotspot.run_hotspot(
    adata=adata_filtered,
    layer_key=GENE_DETECTION_FILTERING["layer_key"],
    latent_obsm_key=hotspot_latent_key,
    umi_counts_obs_key=umi_counts_obs_key,
    model=model,
    n_neighbors=n_neighbors,
    weighted_graph=weighted_graph,
    jobs=jobs,
    top_n=top_n,
    gene_selection_method=gene_selection_method,
    autocorr_fdr_threshold=autocorr_fdr_threshold,
    min_gene_threshold=min_gene_threshold,
    core_only=core_only,
    module_fdr_threshold=module_fdr_threshold,
    compute_local_correlations=compute_local_correlations,
    compute_modules=compute_modules,
    compute_module_scores=compute_module_scores,
    module_prefix=module_prefix,
)

_________

## 10 - Locally variable genes and modules
<hr style="border:2px solid black"> </hr>

This section shows what Hotspot found: which genes carry structure across the
cell neighbourhood graph, which of those move together, and how the resulting
gene programs are distributed over the cells.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# FDR threshold used to filter enrichment results.
# Only enriched terms with an adjusted p-value below this threshold are kept.
fdr_threshold = 0.05

In [ ]:
summary = hotspot.show_hotspot_results(
    adata=adata_filtered,
    fdr_threshold=fdr_threshold
)

# Needs the Hotspot object and can therefore not be read from adata
hotspot.plot_local_correlations(results["hotspot"])

with pd.option_context("display.max.rows", 10, "display.max.columns", None):
    display(summary["autocorrelations"])
    display(summary["module_sizes"])

## 11 - Module enrichment
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Gene set enrichment settings

# Gene set libraries used for enrichment analysis.
# Each Hotspot module will be tested against these gene set collections.
gene_sets = [
    "GO_Biological_Process_2018"
]

# Organism used for the enrichment database.
# Make sure this matches the gene identifiers used in the dataset.
organism = ""

# Minimum number of genes required in a Hotspot module
# before enrichment analysis is performed for that module.
min_module_size = 5

# If True, overwrite existing enrichment results with the same key/output name.
overwrite = True

# File name used to save the enrichment result table.
# Set to None if no table should be saved.
save_table = "hotspot_module_enrichment.tsv"

# Plot enrichment settings

# Number of top enriched terms shown per module in the enrichment plot.
enrichment_top_n = 5

### 11.1 - Run module enrichment

In [ ]:
hotspot.run_hotspot_module_enrichment(
    adata=adata_filtered,
    gene_sets=gene_sets,
    organism=organism,
    min_module_size=min_module_size,
    overwrite=overwrite,
    save_table=save_table,
)

### 11.2 - plot module enrichment

In [ ]:
top_terms = hotspot.plot_hotspot_module_enrichment(
    adata_filtered,
    top_n=enrichment_top_n,
    save="hotspot_module_enrichment.pdf",
    report="hotspot_module_enrichment.png",
)

with pd.option_context("display.max.rows", 100, "display.max.columns", None):
    display(top_terms[["module", "Gene_set", "Term", "Adjusted P-value", "Overlap"]])

## 12 - Save filtered adata
<hr style="border:2px solid black"> </hr>
Store the final results

In [ ]:
# Saving the data
utils.adata.save_h5ad(
    adata_filtered,
    "anndata_hotspot.h5ad",
    report=["hotspot_overview.md", "hotspot_obs.png", "hotspot_var.png"],
)

In [ ]:
sctoolbox.settings.close_logfile()